### **엔트로픽 API와 MCP 서버 활용하기**
- Claude Desktop은 하나의 완성형 서비스로 개발자들이 커스터마이징된 에이전트를 만들기에는 제약이 있음
- API를 활용해보자

#### 동기(Synchronous)방식 vs 비동기(Asynchronous) 방식
- `동기방식`: 하나의 작업이 완전히 끝날 때까지 다음 작업을 시작하지 않고 기다리는 방식
    - (`계산이 복잡하고 순서가 절대적으로 중요한 작업`은 동기방식이 빠르고 안전하며 디버깅이 쉬움)
- `비동기방식`: 작업 A를 시키고, 바로 B작업을 하러 가는 방식
    - (앱이 직접 작업해야 하는게 아니라 `외부 서버가 일하는 것을 기다려야 할 때` 매우 효율적)
- 우리가 만든 앱에서 외부에 있는 MCP 서버에게 요청을 했을 때, 시간이 오래 걸리는 작업이라면 응답을 받기 전에 앱 내 다른 서비스를 제공할 수 없어짐 (비동기 방식을 사용하면 MCP 서버의 응답을 기다리며 로딩 화면 UI를 띄워주거나 다른 도구를 준비하거나 다른 사용자가 채팅을 입력하는지 확인하는 등 여러 작업을 효율적으로 수행할 수 있음)
- 이제부터 API를 활용하여 직접 앱을 제작하려고 하기 때문에, 비동기 방식에 맞는 코드로 작성하는 법을 익혀야 함

In [1]:
!pip install langchain-mcp-adapters langchain-anthropic

   ---------------------------------------- 0.0/621.2 kB ? eta -:--:--
   ---------------------------------------- 621.2/621.2 kB 7.8 MB/s  0:00:00

   ---------------------------------------- 0/3 [anthropic]
   ---------------------------------------- 0/3 [anthropic]
   ---------------------------------------- 0/3 [anthropic]
   ---------------------------------------- 0/3 [anthropic]
   ---------------------------------------- 0/3 [anthropic]
   ---------------------------------------- 0/3 [anthropic]
   ---------------------------------------- 0/3 [anthropic]
   -------------------------- ------------- 2/3 [langchain-anthropic]
   ---------------------------------------- 3/3 [langchain-anthropic]



In [10]:
%%writefile mcpServer/agent_test.py

import os
from dotenv import load_dotenv

# asyncio : 파이썬 비동기 작업 지원 모듈
import asyncio
# ChatAnthropic : 엔트로픽 채팅 모델 클래스
from langchain_anthropic import ChatAnthropic
# load_mcp_tools: langchain 지원 MCP 서버 도구 활용 클래스
from langchain_mcp_adapters.tools import load_mcp_tools
# create_agent: 에이전트 객체 생성 클래스 (내부에 langchain 및 langgraph 코드가 포함됨)
from langchain.agents import create_agent
# StudioServerParameters : MCP 서버 실행 설정값을 담아주는 클래스
from mcp import StdioServerParameters

# stdio_client : MCP 주소와 '로컬서버'를 연결시키는 클래스 ('웹서버'와의 연결은 sse_client를 사용)
from mcp.client.stdio import stdio_client
# ClientSession : 파이썬 명령을 MCP 전용 명령으로 전송하고 응답받는 세션 클래스
from mcp.client.session import ClientSession


load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# async def: 비동기 로직 구현이 가능한 함수로 선언
async def main() :
    print("▶ 1단계: fetch MCP 서버의 주소 세팅")
    server_params = StdioServerParameters(
        command="C:/Users/agnes/anaconda3/Scripts/uvx.exe",
        args=["mcp-server-fetch"]  # fetch 서버 명칭
    )

    print("▶ 2단계: 서버에 통신망(Session) 연결")
    # 아래 코드는 로컬 MCP 서버와 API 서버를 모두 사용하는 에이전트 구현에서 필수적인 국제 표준
     # async로 대기시간이 발생하는 통신 코드에 비동기 실행이 가능하게 설정
      # 해당 MCP 서버를 실행하고 데이터 입력(read) 및 출력(write) 객체를 지정
      # write는 파이썬이 MCP 서버에 요청하는 통로, read는 MCP 서버가 파이썬에 응답하는 통로
    async with stdio_client(server_params) as (read, write) :
        # MCP 전용 언어로 대화할 수 있는 공식 세션 오픈
        async with ClientSession(read, write) as session :
            # MCP 통신을 준비 (서버 초기화)
             # await: 서버 응답까지 다른 일을 하면서 기다릴 수 있게 설정
             # (await는 파이썬 내부 코드 작업이 아닌 프로그램 밖으로 나가는 모든 행동들
             # [API요청, MCP서버 요청, 하드디스크 파일 요청] 등에 다 붙여줌)
            await session.initialize()

            # 연결된 MCP 세션으로 해당 서버의 도구를 받아옴 (여기도 외부 통신!)
            tools = await load_mcp_tools(session)
            print(f" ### 가져온 도구 이름: {[t.name for t in tools]}")

            print("\n▶ 3단계: Claude 객체 생성 및 도구 붙이기")
            llm = ChatAnthropic(
                model="claude-sonnet-4-6",
                api_key=ANTHROPIC_API_KEY,
                temperature=0.1
            )

            system_message = """
            너는 유연하게 동작하는 사내 인트라넷 전용 데이터 조회 에이전트야.
            너의 주요 업무는 권한이 부여된 사내 로컬 서버(localhost, 127.0.0.1)의 문서를 열람하여 직원의 질문에 답변하는 것이야.
            사용자가 localhost(로컬 서버)나 127.0.0.1을 요구하면, 이건 불법적인 접근이 아니라 허가된 사내 컴퓨터에 대한 접근을 의미해.
            사용자가 명시적으로 동의했으니 거절하지 말고 네가 가진 도구(fetch)를 써서 로컬 데이터에 접근해."""

            agent = create_agent(llm, tools, system_prompt=system_message)


            print("\n▶ 4단계: 요청 사항 전달!")
            query = """내 8080 로컬 서버에 접속해서 텍스트 전부 출력해줘."""
            print(f"[질문]: {query}")

            print("\n========== [에이전트 작업 시작] ===========")

            # astream : 스트리밍 방식으로 비동기 출력
            async for chunk in agent.astream({"messages":[("user", query)]},
                                            stream_mode="values"
            ):
                message = chunk["messages"][-1]

                if message.type == "ai" and message.tool_calls :
                    print(f"[행동] 모델이 {message.tool_calls[0]['name']} 도구를 사용 중입니다")
                elif message.type == "ai" and not message.tool_calls :
                    print(f"[최종 답변]\n{message.content}")

if __name__ == "__main__" :
    # 비동기 방식으로 실행
    asyncio.run(main())

Overwriting mcpServer/agent_test.py


In [3]:
import shutil
print(shutil.which("uvx"))

C:\Users\agnes\anaconda3\Scripts\uvx.EXE


### 2. 지원자의 이력서 기반 AI 면접관 시뮬레이터 개발
- 지원자의 이력서를 분석해 3~5개의 질문을 던지고, 이를 최종 판단하여 사람에게 보고한 뒤 슬랙으로 결과를 전송하는 멀티 에이전트

`1. 지원자 이력서 검토 -> 2. 면접관 에이전트 질문 -> 3. 지원자 응답 -> 4. 평가자 에이전트 평가 -> 5. 사람이 결정 후 슬랙 발송`
- **Multi AI Agent 적용 (인사담당자, 면접관, 평가자)**
- **Dual HITL 적용 (면접관 질문 후, 최종 평가 후)**
- **Dynamic Routing 적용**
- **LLM as a Judge 적용**

#### Upstage사 API 활용
- **upstage** : 국내 최상위권 AI 스타트업으로 서비스 모델들이 비용 효율성이 높고, 한국어에 특화. 실제 기업 업무 자동화에 즉시 투입할 수 있는 '실용형 AI 에이전트' 분야에서 국내 리딩 기업 위치에 있음
- Upstage Console 사이트: https://console.upstage.ai/docs/getting-started
- 26년 3월 출시된 대표모델 Solar-Pro3의 경우 한국어 특화 성능이 gpt4o 이상으로 알려져 있음. API 사용 금액도 1/15 수준.
- 주요 API로 `채팅`, `문서파싱`, `정보추출`, `문서분류` 등이 있음
- 특히 문서파싱 API는 **비전 AI 기반 문서 레이아웃 분석**(구조를 픽셀 단위로 분석) + **고성능 OCR**(광학 문자 인식) 기능이 포함되어 있어 복잡한 문서에서 표나 다단글 구조를 완벽에 가깝게 유지. RAG나 정보추출 에이전트를 만들 때 성능이 좋음.

In [1]:
!pip install -q langchain-upstage

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0rc1 requires huggingface-hub<2.0,>=1.2.1, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.0.0rc1 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.


In [ ]:
%%writefile mcpServer/interview_agent.py

import os
from dotenv import load_dotenv
import asyncio

# List: 리스트  객체 생성 (리스트에 특정 자료형만 들어오게 설정 가능)
# Literal : 반환 문자열 고정 클래스 (변수나 함수의 반환값이 지정한 단어/문자열들 중 하나만 나와야 하는 경우 사용)
from typing import Annotated, TypedDict, List, Literal

### pydantic : LLM의 자연어 응답을 정확한 양식(JSON)으로 정제해주는 라이브러리
 # BaseModel : 클래스 생성시 데이터 유효성을 검사하는 클래스
 # (langchain은 BaseModel을 상속받은 클래스의 구조를 분석하여 LLM이 이해할 수 있는 JSON형식으로 자동 변환)
 # Field : BaseModel 내부의 각 속성에 대해 추가적인 메타데이터를 제공하고 데이터 제약 조건을 세밀하게, 유연하게 설정
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic

# Upstage 문서 파싱 API 클래스 (일반 문서 로더들보다 성능이 좋음)
from langchain_upstage import UpstageDocumentParseLoader

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

# ====================================================
# 1. 상태(state) 및 각 에이전트 출력 형식 정의
# ====================================================

# 1) 상태 정의
class InterviewState(TypedDict) :
    messages: Annotated[list, add_messages]
    applicant_name: str   # 지원자 이름
    resume_content: str   # 이력서 내용
    keywords: List[str]   # 핵심 키워드 (문자열만 받는 리스트로 정의)
    question_count: int    # 질문 횟수
    is_finished: bool    # 면접관과 에이전트가 스스로 판단하는 종료 플래그
    evaluator_result: str    # 평가 결과
    evaluator_reason: str    # 평가 이유
    final_decision: str     # 인사담당자의 최종 승인 플래그

# 2) 인사담당자 에이전트용 출력양식 지정 (BaseModel 클래스 상속)
class HRAnalysis(BaseModel) :
    applicant_name: str = Field(description="이력서에서 추출한 지원자 이름")
    keywords: List[str] = Field(description="면접에 활용할 수 있는 핵심 기술/경험 키워드 3~5개")

# 3) 면접관 에이전트용 출력양식 지정
class InterviewerAction(BaseModel) :
    question: str = Field(default="", description="지원자에게 던질 꼬리 질문 (종료시 빈 문자열)")
    # 면접을 종료할 때는 질문을 더 만들지 않도록 default값으로 빈 문자열 부여
    # (pydantic으로 변수를 선언했는데 사용하지 않으면 에러가 발생하기 때문)
    is_finished: bool = Field(description="충분히 검증되어 질문을 그만하고 평가자에게 넘길거면 True, 아니면 False")

# 4) 평가자 에이전트용 출력양식 지정
class EvaluatorResult(BaseModel):
    status: Literal["PASS", "FAIL"] = Field(description="최종판정 (반드시 PASS 또는 FAIL)")
    reasoning: str = Field(description="합격/불합격으로 판단한 결정적 사유 3문장 이내")


# ====================================================
# 2. 함수 & LLM 세팅
# ====================================================
def send_final_email(applicant_name:str, result:str) -> None :
    """최종 결정에 따라 합/불 안내 이메일을 발송합니다."""
    print(f"\n[메일 시스템 가동] 수신자: {applicant_name} 지원자님")
    if result == "PASS" :
        print("내용: 축하합니다! AI 면접 전형에 '최종합격'하셨습니다. \n")
    else :
        print("내용: 안타깝게도 이번 전형에서는 모시지 못하게 되었습니다.")

llm = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0.2,
    api_key=ANTHROPIC_API_KEY
)


# ====================================================
# 3. 노드 정의
# ====================================================

# 1) HR 시작 노드 (면접을 처음 시작할 때)
async def hr_start_node(state:InterviewState) :
    print("[HR 담당자] Upstage API로 이력서를 분석합니다...")

    file_path = "mcpServer/김민정_DBA_이력서.pdf"
    loader = UpstageDocumentParseLoader(
        file_path, api_key=UPSTAGE_API_KEY,
        # 로드한 문서를 분리할 단위 설정
         # "None" : 문서 전체를 하나의 Document 객체로 반환
         # "element" : 의미있는 레이아웃 요소 (문단, 표, 제목 등)으로 반환
         # "page"
        split='element',
        # 출력 텍스트 형식 지정
         # "html" : HTML 태그로 감싸서 반환
         # "text" : 순수 텍스트로 바환
         # "markdown" (LLM이 이해하기 쉬움)
        output_format="markdown"
    )

    resume_text = "\n".join([doc.page_content for doc in loader.load()])

    prompt = f"""다음 이력서를 분석하여 지원자 이름과 면접용 핵심 키워드를 추출하세요.
    [이력서]
    {resume_text}
    """

    # analysis 변수에 HRAnalysis 클래스에 정의된 규격에 맞는 데이터만 들어가도록 함
     # with_structured_output : 특정 에이전트의 JSON 형식에 맞춰서만 응답하도록 지시
     # ainvoke : 비동기 응답 실행
    analysis: HRAnalysis = await llm.with_structured_output(HRAnalysis).ainvoke(
        [HumanMessage(content=prompt)]
    )

    # 실제로 analysis 변수에는 아래 형태로 HRAnalysis 객체 자체가 저장됨
    # HRAnalysis(applicant_name="김민정", keywords=["python", "Langchain"])

    print(f" -> [추출 완료] 지원자: {analysis.applicant_name} / 키워드: {analysis.keywords}")

    return {
        "applicant_name": analysis.applicant_name,   # 이력서 검토 후 추출한 지원자 이름
        "resume_content": resume_text,
        "keywords": analysis.keywords,
        "question_count": 0,                      # 질문 횟수 (아직 질문 전이므로 0)
        "is_finished":False                     # 질문 마무리 플래그 (아직 질문 전)
        }

# 2) 면접관 노드
async def interviewr_node(state: InterviewState) :
    # TypedDict를 상속받은 state이므로 거대한 딕셔너리라고 보면 됨
    count = state.get("question_count", 0)     # 있으면 가져오고, 없으면 0 값
    
    # 질문은 최대 5개까지
    if count >= 5 :
        print(" -> [면접관] 최대 질문 횟수에 도달하여 면접을 종료합니다.")
        return {"is_finished": True}
    
    print(f"\n[면접관] {count+1} 번째 질문을 고민 중입니다...")

    sys_msg = SystemMessage(content=f"""당신은 친절한 기술 면접관입니다.
    지원자: {state['applicant_name']} / 핵심 키워드: {state['keyword']}
    
    [진행 규칙]
    - 현재까지 당신이 던진 질문 횟수: {count}회
    - 질문이 3회 미만이면 검증이 부족하므로 '무조건' 꼬리질문이나 다른 질문을 던지세요. (is_finished=False)
    - 질문이 '3회 이상~4회 이하'라면, 답변이 충분한지 스스로 판단하여 조기종료(is_finished=True)하거나 질문을 5회까지 이어갈 수 있습니다.
    """)

    # 만약 대화 기록이 텅 비어있다면 (첫번째 질문 전)
    chat_history = state["message"]
    if not chat_history :
        # Claude 모델은 대화 시작시 무조건 1개의 HumanMessage가 있어야 동작함
        # openai 모델은 systemprompt만 있어도 되는디
        chat_history = [HumanMessage(content="면접을 시작하고 첫 번째 질문을 던져주세요.")]

    action: InterviewerAction = await llm.with_structured_output(InterviewerAction).ainvoke(
        [sys_msg] + chat_history
    )

    # 3회 미만인데 LLM이 마음대로 종료하려 하면 강제취소
    if action.is_finished and count < 3 :
        action.is_finished = False
    
    # 조기 종료시
    if action.is_finished :
        print(" -> [면접관] 충분한 검증이 완료되어 면접을 조기 종료합니다.")
        return {"is_finished": True}
    # 질문 이어갈 시
    else :
        print(f" -> [질문]: {action.question}")
        return {
            "messages" : [AIMessage(content=action.question)],
            "question_count" : count+1,
            "is_finished": False
        }


# 3) 평가자 노드
async def evaluator_node(state: InterviewState) :
    print('\n[평가자] 질의응답 전체 기록을 분석하여 최종 합/불을 판단합니다...')
    prompt = f"""당신은 냉철한 평가자 입니다. 다음 대화 기록을 보고 지원자의 역량을 평가하세요.
    반드시 PASS 또는 FAIL 중 하나만 선택하고, 사유를 작성하세요.
    """

    result: EvaluatorResult = await llm.with_structured_output(EvaluatorResult).ainvoke(
        state['messages'] + [HumanMessage(content=prompt)]
    )

    print(f" -> [AI 평가 결과]: {result.status} (사유: {result.reasoning})")

    return {
        "evaluator_result": result.status,
        "evaluator_reason": result.reasoning
    }

# 4) HR 마무리 노드 (이메일은 그냥 내용 출력만!)
async def hr_end_node(state: InterviewState) :
    print("\n[HR 담당자] 인사권자의 최종 결정을 접수하여 이메일을 발송합니다.")
    send_final_email(state['applicant_name'], state['final_decision'])
    return {}


# ====================================================
# 4. 조건부 라우팅 함수
# ====================================================

def route_after_interview(state: InterviewState):
    # 질문 끝낸다고 판단하면 -> 평가자
    if state.get("is_finished", False) == True :
        return "evaluator"
    # 아니면 다시 자기 -> 면접관
    return "interviewer"

# 그래프 및 노드 구성
graph = StateGraph(InterviewState)
graph.add_node('hr_start', hr_statr_node)
graph.add_node('interviewer', interviewer_node)
graph.add_node('evaluator', evaluator_node)
graph.add_node('hr_end', hr_end_node)

graph.add_edge(START, 'hr_start')
graph.add_edge('hr_start', 'interviewer')
graph.add_conditional_edges('interviewer', route_after_interview)
graph.add_edge('evaluator', 'hr_end')
graph.add_edge('hr_end', END)

app = graph.compile(checkpointer=MemorySaver(),
                    # interrupt_after: 특정 노드가 실행된 후 멈추도록 설정 (듀얼 HITL 적용)
                    interrupt_after=['interviewer', 'evaluator'])


# ====================================================
# 5. 메인 루프
# ====================================================
async def main() :
    config = {"configurable": {"thread_id": "user1"}}
    initial_state = {"messages": []}

    print("=====================================")
    print("[HR 시스템] AI 면접 시스템 시작")
    print("=====================================\n")

    # 노드에 이미 출력문이 다 있기 때문에 app 객체가 스트리밍으로만 출력하도록 pass 설정
    async for _ in app.astream(initial_state, config, stream_mode="updates") : pass

    while True :
        # 상태 가져와서 다음 실행될 노드 확인
        snapshot = app.get_state(config)
        next_step = snapshot.next

        # 다음 실행될 노드가 없다면 (END 노드는 속이 비어있음)
        if not next_step :
            print("\n### 모든 채용 프로세스가 종료되었습니다.")
            break
        
        # 면접관 노드라면
        if next_step[0] == 'interviewer' :
            user_input = input("\n 지원자 답변 (포기 'q'): ")
            if user_input.lower() == 'q' :
                break

            # state의 messages에 지원자 답변을 업데이트
            app.update_state(config,
                            {"messages": [HumanMessage(content=user_input)]}
                            )

        # 평가자 노드라면
        # (면접관 노드가 실행을 끝내면 is_finished가 True가 되어 평가자로 넘어간 상황)
        elif next_step[0] == 'evaluator' :
            print("\n 질문이 끝났습니다. 데이터를 평가자에게 넘깁니다...")
        

        # hr_end 라면
        elif next_step[0] == 'hr_end' :
            print("\n" + "="*50)
            print("### [인사팀장 결재 대기] ###")
            print(f"지원자: {snapshot.values['applicant_name']}")
            print(f"AI 평가: [{snapshot.values['evaluator_result']}] ({snapshot.values['evaluator_reason']})")
            print("="*50)

            final_choice = input(" ▶ 최종 결정을 입력하세요 (PASS/FAIL):")
            app.update_state(config, {'final_decision': final_choice.upper()})
        
        # stream 형식 비동기 출력
        async for _ in app.astream(None, config, stream_mode="updates") : pass

if __name__ == "__main__" :
    asyncio.run(main())

Overwriting mcpServer/interview_agent.py


### 3. AI 면접관 시뮬레이터 & MCP 결합
- 사용자의 기술적 답변을 웹에서 최신으로 검색해서 판단할 수 있는 Tavily Search MCP 서버 추가
- 합,불을 최종 판단하면 Slack에 합,불에 대한 결과를 전송하도록 Slack MCP 서버 추가
  - Tavily와 Slack의 MCP 서버를 직접 사용하려면 Node.js기반의 npx 사용이 필요함
  - Node.js 설치 사이트 : https://nodejs.org/ko
  - 사이트 상단 '다운로드' 탭에서 OS에 맞게 설치 프로그램 다운받아 실행(설치 중 체크화면에는 체크하지 말 것)
  - 설치 후에는 쥬피터노트북을 완전히 껐다 켜야 자동 인식됨

In [7]:
import shutil
print(shutil.which('npx'))

C:\Program Files\nodejs\npx.CMD


In [2]:
!pip freeze > my_backup.txt

In [5]:
!pip install --user langchain-upstage "huggingface-hub>=1.2.1,<2.0" "tokenizers>=0.22.0,<=0.23.0"

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
INFO: pip is looking at multiple versions of langchain-upstage to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_upstage-0.7.6-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_upstage-0.7.5-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_upstage-0.7.4-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-0.3.84-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_upstage-0.7.3-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_upstage-0.7.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_upstage-0.7.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_upstage-0.7.0-py3-none-any.whl.metadata (3.3 kB)
INFO: pip is still looking at multiple versions of langchain-upstage to determine which version is compatib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.2.43 which is incompatible.
langchain-cohere 0.5.0 requires langchain-core<2.0,>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.


In [5]:
!pip install langchain-anthropic

  Using cached langchain_core-1.2.28-py3-none-any.whl.metadata (4.4 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.1.147

    Uninstalling langsmith-0.1.147:

      Successfully uninstalled langsmith-0.1.147

   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
  Attempting uninstall: langchain-core
   ---------------------------------------- 0/2 [langsmith]
    Found existing installation: langchain-core 0.2.43
   ---------------------------------------- 0/2 [langsmith]
   -------------------- ------------------- 1/2 [langchain-core]
    Uninstalling langchain-core-0.2.43:
   -------------------- ------------------- 1/2 [langchain-core]
      Successfully uninstalled langchain-core-0.2.43
   -------------------- ------------------- 1/2 [langchain-core]
   -------------------- ------------------- 1/2 [langchain-core]
   ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.1.25 requires langchain-core<0.3.0,>=0.2.40, but you have langchain-core 1.2.28 which is incompatible.
langchain-upstage 0.1.7 requires langchain-core<0.3,>=0.2.2, but you have langchain-core 1.2.28 which is incompatible.


In [6]:
pip install "langchain-core>=0.2.40,<0.3.0" langchain-anthropic langchain-openai langchain-upstage --upgrade

  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_upstage-0.7.7-py3-none-any.whl.metadata (3.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of langchain-anthropic to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_anthropic-1.4.0-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is still looking at multiple versions of langchain-anthropic to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [48 lines of output]
      Checking for Rust toolchain....
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp313-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\agnes\AppData\Local\puccinialin\puccinialin\Cache
      
      Installing rust to C:\Users\agnes\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\agnes\.rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: default host triple is aarch64-pc-windows-msvc
      info: syncing cha